# ARKITEKT — ComfyUI grátis no Colab

Sobe um servidor ComfyUI com Flux.1-dev + ControlNet Union (depth) numa GPU
grátis do Colab e expõe uma URL pública temporária. Use essa URL como
`ARKITEKT_COMFY_URL` no `.env` local ou nos secrets do app Streamlit.

**Antes de rodar:**
- Ative a GPU: menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
- Crie um token em https://huggingface.co/settings/tokens e aceite a licença em
  https://huggingface.co/black-forest-labs/FLUX.1-dev (o modelo é *gated*, sem isso o download falha).

**Limites do plano grátis:** sessão cai depois de ~12h (ou bem antes, por
ociosidade), disco é apagado ao encerrar, e a URL do túnel muda a cada vez
que você reinicia este notebook. Para uso esporádico de teste, tudo bem —
não é uma URL fixa de produção. Rode a Célula 6 (Google Drive) se quiser
evitar rebaixar ~25 GB de modelos toda sessão.

## 1. Checar GPU

In [ ]:
!nvidia-smi

## 2. Instalar ComfyUI

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

## 3. Nós extras — preprocessador de depth (Depth Anything V2)

Necessário para gerar o mapa de profundidade a partir do screenshot dentro
do próprio ComfyUI (a rota fal.ai faz isso num endpoint separado; aqui
precisamos do nó equivalente).

In [ ]:
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux
%cd comfyui_controlnet_aux
!pip install -q -r requirements.txt
%cd /content/ComfyUI

## 4. Login no Hugging Face (FLUX.1-dev é *gated*)

Cole o token quando pedir — gere em https://huggingface.co/settings/tokens
e antes disso aceite a licença em https://huggingface.co/black-forest-labs/FLUX.1-dev.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Baixar modelos (~25 GB — leva alguns minutos)

Variantes fp8 para caber na GPU de 16 GB do Colab grátis (T4).

In [ ]:
import os
os.makedirs("models/unet", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/clip", exist_ok=True)
os.makedirs("models/controlnet", exist_ok=True)

!wget -q --show-progress -O models/unet/flux1-dev-fp8.safetensors \
    https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors
!wget -q --show-progress -O models/vae/ae.safetensors \
    https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors
!wget -q --show-progress -O models/clip/clip_l.safetensors \
    https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors
!wget -q --show-progress -O models/clip/t5xxl_fp8_e4m3fn.safetensors \
    https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors
!wget -q --show-progress -O models/controlnet/flux-controlnet-union.safetensors \
    https://huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union/resolve/main/diffusion_pytorch_model.safetensors
print("modelos prontos")

## 6. (Opcional) Google Drive — evita rebaixar os modelos toda sessão

Rode isto **antes** da célula 5 numa próxima sessão, e ela vai pular os
downloads já feitos (o wget não sobrescreve se você trocar por `-nc`,
ou simplesmente aponte `models/` para uma pasta no Drive symlinkada).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
cache = pathlib.Path('/content/drive/MyDrive/arkitekt_comfy_models')
cache.mkdir(parents=True, exist_ok=True)
local = pathlib.Path('/content/ComfyUI/models')
for sub in ['unet', 'vae', 'clip', 'controlnet']:
    (cache / sub).mkdir(exist_ok=True)
    if (local / sub).is_dir() and not (local / sub).is_symlink():
        for f in (local / sub).iterdir():
            f.rename(cache / sub / f.name)
        (local / sub).rmdir()
    if not (local / sub).exists():
        (local / sub).symlink_to(cache / sub)
print("models/ agora aponta para o Drive — rode a célula 5 de novo se faltar algo")

## 7. Subir o servidor ComfyUI em background

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(15)
print("ComfyUI subindo — confira a célula 8 para a URL pública")

## 8. Expor com túnel público (cloudflared — sem cadastro)

Copie a URL `https://xxxx.trycloudflare.com` impressa abaixo. Ela serve
tanto pra abrir a interface do ComfyUI no navegador (montar o workflow,
próximo passo) quanto pra `ARKITEKT_COMFY_URL` do ARKITEKT.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        print('\n>>> copie a URL acima (https://....trycloudflare.com) <<<')
        break

## 9. Próximo passo

1. Abra a URL do túnel no seu navegador — é a interface do ComfyUI.
2. Monte o workflow uma única vez seguindo `docs/comfyui_gratis.md` do repo
   ARKITEKT (lista exata de nós e como conectá-los) e exporte em
   **Save (API Format)** como `flux_depth.json`.
3. Coloque o arquivo em `workflows/flux_depth.json` no repo e confira os
   ids em `core/engines/comfy_engine.py` (dict `NODE`).
4. Exporte `ARKITEKT_COMFY_URL` (ou cole no app Streamlit / secrets) com a
   URL do túnel e rode `python bench/run.py --engines comfy` ou o app.

Mantenha esta aba do Colab aberta — fechar encerra o servidor e derruba a URL.